# 03 · Model Training

Trains three complementary models from the feature table:

| Model | Task | Algorithm |
|---|---|---|
| **LightGBM Forecaster** | Predict solar/wind t+6h, t+12h, t+24h | LightGBM (CPU multi-core) |
| **LSTM Autoencoder** | Anomaly detection via reconstruction error | PyTorch LSTM (CUDA) |
| **Mismatch Classifier** | 4-class supply-demand severity | XGBoost (GPU) |

**Outputs**: model artefacts in `models/`

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from pathlib import Path
from src.models import MODELS_DIR, PROCESSED_DIR, DEVICE

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Feature parquet: {(PROCESSED_DIR / "features.parquet").exists()}')

In [ ]:
# ── 1. LightGBM Renewable Forecaster ─────────────────────────────────────────
from src.models import train_lgbm

lgbm_metrics = train_lgbm()
print('\nLightGBM metrics summary:')
for model_name, m in lgbm_metrics.items():
    print(f'  {model_name:15s}  MAE={m["mae"]:7.2f} MW  RMSE={m["rmse"]:7.2f} MW  R²={m["r2"]:.4f}')

In [ ]:
# ── 2. LSTM Autoencoder ───────────────────────────────────────────────────────
from src.models import train_lstm

train_lstm()

In [ ]:
# Plot LSTM training curves
import joblib
import matplotlib.pyplot as plt

history = joblib.load(MODELS_DIR / 'lstm_history.pkl')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(history['epoch'], history['train_loss'], label='Train loss')
ax.plot(history['epoch'], history['val_loss'],   label='Val loss', linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.set_title('LSTM Autoencoder — Training Curves')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 3. XGBoost Mismatch Classifier ───────────────────────────────────────────
from src.models import train_xgboost

xgb_metrics = train_xgboost()
print('\nXGBoost best iteration:', xgb_metrics['best_iteration'])

In [ ]:
# List all saved artefacts
print('Saved model artefacts:')
for f in sorted(MODELS_DIR.iterdir()):
    size = f.stat().st_size / 1e6
    print(f'  {f.name:45s}  {size:.2f} MB')